In [1]:
import pandas as pd
import requests
import numpy as np

In [2]:
# URLs for All-NBA Teams
urls = {
    "1st Team": "https://www.espn.com/nba/history/awards/_/id/44",
    "2nd Team": "https://www.espn.com/nba/history/awards/_/id/45",
    "3rd Team": "https://www.espn.com/nba/history/awards/_/id/46"
}

In [ ]:
import pandas as pd
import requests
import numpy as np

# URLs for All-NBA Teams
urls = {
    "1st Team": "https://www.espn.com/nba/history/awards/_/id/44",
    "2nd Team": "https://www.espn.com/nba/history/awards/_/id/45",
    "3rd Team": "https://www.espn.com/nba/history/awards/_/id/46"
}

all_data = []

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
}

for team_type, url in urls.items():
    print(f"Scraping {team_type} from {url}...")
    try:
        response = requests.get(url, headers=headers)
        if response.status_code == 200:
            dfs = pd.read_html(response.content)
            print(f"  Found {len(dfs)} tables.")
            
            target_df = None
            # Heuristic: The table we want is likely the largest one (most rows)
            if dfs:
                target_df = max(dfs, key=lambda x: x.shape[0])
                print(f"  Largest table shape: {target_df.shape}")

            if target_df is not None and target_df.shape[0] > 5:
                # Fix headers
                # Check if first row looks like headers (contains 'Year' or 'Player')
                header_row_idx = None
                for i in range(min(5, len(target_df))):
                    row_str = target_df.iloc[i].astype(str).str.upper().tolist()
                    if 'PLAYER' in row_str or 'YEAR' in row_str or 'SEASON' in row_str:
                        header_row_idx = i
                        break
                
                if header_row_idx is not None:
                    # Set header
                    target_df.columns = target_df.iloc[header_row_idx]
                    target_df = target_df[header_row_idx + 1:]
                
                # Normalize columns
                target_df.columns = [str(c).upper().strip() for c in target_df.columns]
                
                if 'PLAYER' in target_df.columns:
                    # Filter out repeated header rows
                    target_df = target_df[target_df['PLAYER'] != 'PLAYER']
                    
                    # Handle Year Column
                    year_col = 'YEAR' if 'YEAR' in target_df.columns else 'SEASON' if 'SEASON' in target_df.columns else None
                    
                    if year_col:
                        # Forward fill year - replacing empty strings or nulls
                        target_df[year_col] = target_df[year_col].replace('', np.nan).ffill()
                        
                        # Select columns
                        cols_to_keep = [year_col, 'PLAYER']
                        if 'POS' in target_df.columns:
                            cols_to_keep.append('POS')
                        if 'TEAM' in target_df.columns:
                            cols_to_keep.append('TEAM') # Capture Team
                        
                        subset = target_df[cols_to_keep].copy()
                        subset = subset.rename(columns={year_col: 'Year', 'POS': 'Position', 'PLAYER': 'Player', 'TEAM': 'Team'})
                        subset['Team_Type'] = team_type
                        
                        all_data.append(subset)
                    else:
                         print(f"  Could not find Year/Season column in table for {team_type}")
                else:
                    print(f"  Could not find 'PLAYER' column. Columns found: {target_df.columns.tolist()}")

            else:
                print(f"  No suitable large table found for {team_type}")
        else:
            print(f"Failed to fetch {url}: {response.status_code}")
            
    except Exception as e:
        print(f"Error scraping {team_type}: {e}")

# Combine and Process
if all_data:
    final_df = pd.concat(all_data, ignore_index=True)
    
    def clean_year(val):
        try:
            val = str(val).strip()
            # Handle "2023-24" or just "2025"
            if '-' in val:
                 parts = val.split('-')
                 if len(parts[0]) == 4:
                     return int(parts[0]) + 1 if len(parts[1]) > 0 else int(parts[0])
            return int(float(val))
        except:
            return None

    final_df['Year'] = final_df['Year'].apply(clean_year)
    
    # Drop rows without player or year
    final_df = final_df.dropna(subset=['Player', 'Year'])
    final_df['Year'] = final_df['Year'].astype(int)
    
    # FILTER: Only keep years 2015-2026
    final_df = final_df[(final_df['Year'] >= 2015) & (final_df['Year'] <= 2026)]
    
    # Sort
    final_df = final_df.sort_values(by=['Year', 'Team_Type', 'Player'], ascending=[False, True, True])
    
    # Final Selection
    cols = ['Year', 'Player', 'Team', 'Team_Type', 'Position'] # Added Team
    
    # Ensure columns exist
    if 'Position' not in final_df.columns:
        final_df['Position'] = 'Unknown'
    if 'Team' not in final_df.columns:
        final_df['Team'] = 'Unknown'
    
    df = final_df[cols]
    
    print("\nScraping Complete!")
    print(f"Total rows: {len(df)}")
    print(df.head(10))
    
    df.to_csv('nba_all_nba_teams.csv', index=False)
    print("Saved to nba_all_nba_teams.csv")
    
else:
    print("No data collected.")

Scraping 1st Team from https://www.espn.com/nba/history/awards/_/id/44...
  Found 1 tables.
  Largest table shape: (398, 10)
Scraping 2nd Team from https://www.espn.com/nba/history/awards/_/id/45...
  Found 1 tables.
  Largest table shape: (396, 10)
Scraping 3rd Team from https://www.espn.com/nba/history/awards/_/id/46...
  Found 1 tables.
  Largest table shape: (187, 10)

Scraping Complete!
Total rows: 975
     Year                   Player                    Team Team_Type Position
2    2025         Donovan Mitchell     Cleveland Cavaliers  1st Team       SG
0    2025    Giannis Antetokounmpo         Milwaukee Bucks  1st Team       PF
3    2025             Jayson Tatum          Boston Celtics  1st Team       SF
1    2025             Nikola Jokic          Denver Nuggets  1st Team        C
4    2025  Shai Gilgeous-Alexander   Oklahoma City Thunder  1st Team       PG
400  2025          Anthony Edwards  Minnesota Timberwolves  2nd Team       SG
399  2025              Evan Mobley     Clev

In [ ]:
df = df

In [4]:
# Preview the dataframe
df.head()

,Year,Player,Team,Team_Type,Position
2,2025,Donovan Mitchell,Cleveland Cavaliers,1st Team,SG
0,2025,Giannis Antetokounmpo,Milwaukee Bucks,1st Team,PF
3,2025,Jayson Tatum,Boston Celtics,1st Team,SF
1,2025,Nikola Jokic,Denver Nuggets,1st Team,C
4,2025,Shai Gilgeous-Alexander,Oklahoma City Thunder,1st Team,PG
